# 05 — Scanner & PrimaryQualityScore

**Auteur :** Benoît Moraillon — **Licence :** AGPL-3.0

`kodoneko_scanner` étend `kodoneko_metrics` à l'échelle d'un dépôt entier :

- parcourt les fichiers (`git ls-files` ou `os.walk`)
- détecte automatiquement le langage par extension
- applique les 4 familles de métriques sur chacun
- agrège dans `RepoMetrics` (totaux + détail fichier par fichier)
- calcule un **PrimaryQualityScore** : score qualité 0-100 + grade A-E
- repère les **hotspots** (fichiers les plus complexes)
- accepte une configuration **TOML** pour personnaliser poids et seuils

## Vocabulaire — quality score vs risk (rappel)

KodoNeko utilise deux indicateurs 0-100 d'orientation **opposée**. Ne pas les confondre :

| Indicateur | Orientation | Où le trouver |
|---|---|---|
| **risk** (`UnderstandabilityMetrics.risk`) | *Pire si plus haut* | Niveau de difficulté d'un fichier (notebook 04) |
| **score** (`PrimaryQualityScore.score`) | *Mieux si plus haut* | Note de qualité globale d'un dépôt (ce notebook) |

Le **PrimaryQualityScore** est l'indicateur de **santé** d'un dépôt. Un score de 85 = bon ; un score de 30 = critique. C'est l'inverse exact d'un score de risque.

## PrimaryQualityScore : pondération par défaut

| Métrique | Poids | Rôle |
|---|---|---|
| `understandability` | **70 %** | Composite ISO/IEC 25010, indicateur principal |
| `mccabe` | 15 % | Chemins d'exécution, signal indépendant |
| `ncloc` | 10 % | Taille brute |
| `halstead` | 5 % | Densité d'information (secondaire) |

Ces poids sont **paramétrables via un fichier `kodoneko.toml`** (cf. dernière section).


In [ ]:
# Bootstrap : permet l'import des modules sans installation pip
import sys
from pathlib import Path

root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import setup_paths
setup_paths.setup()

## Préparer un mini-dépôt de test

In [ ]:
import tempfile
from pathlib import Path

# Créer un mini-dépôt avec quelques fichiers de qualité variable
tmp = Path(tempfile.mkdtemp(prefix="kodoneko_demo_"))

(tmp / "propre.py").write_text('''
def somme(a, b):
    \"\"\"Addition basique.\"\"\"
    return a + b

def produit(a, b):
    return a * b
''')

(tmp / "moyen.py").write_text('''
def categoriser(age):
    if age < 13:
        return "enfant"
    elif age < 18:
        return "ado"
    elif age < 65:
        return "adulte"
    return "senior"
''')

(tmp / "hotspot.py").write_text('''
def f(a, b, c, d, e):
    r = 0
    if a:
        for i in range(b):
            if c[i]:
                for j in range(d):
                    if e[j]:
                        r += 1
    return r
''')

print(f"Dépôt de test : {tmp}")
for p in sorted(tmp.iterdir()):
    print(f"  - {p.name}")

## Scanner le dépôt

In [ ]:
from kodoneko_scanner import scan_repo

report = scan_repo(str(tmp), use_git=False)
print(f"Fichiers analysés : {report.files_scanned}")
print(f"NCLOC total       : {report.totals.ncloc}")
print(f"MCcabe max        : {report.totals.mccabe_max}")
print(f"Cognitive max     : {report.totals.cognitive_max}")
print(f"Understandability max : {report.totals.understandability_max:.1f}")

## Calculer le PrimaryQualityScore

In [ ]:
from kodoneko_scanner import compute_primary_quality_score

bs = compute_primary_quality_score(report)
print(f"Quality score : {bs.score:.1f}/100   (plus c'est haut, mieux c'est)")
print(f"Grade         : {bs.grade}")
print(f"Synthèse      : {bs.summary}")
print()
print("Détail par métrique :")
for metric, info in bs.breakdown.items():
    print(f"  {metric:<20} (poids {info['weight']*100:>4.0f}%) "
          f"pénalité moy. = {info['penalty_avg']:>5.1f}")


## Trouver les hotspots

In [ ]:
# Tri par défaut : understandability (depuis v0.4)
hotspots = report.hotspots(top=5, by="understandability")
print("Top 5 hotspots (par risk d'understandability décroissant) :")
for f in hotspots:
    u = f.metrics.understandability
    print(f"  {f.relative_path:<20}  risk={u.risk:>5.1f} ({u.level:<10})  cognitive={u.cognitive}")

print()
# Tri par McCabe pour comparer
hotspots_mc = report.hotspots(top=5, by="mccabe")
print("Top 5 hotspots (par McCabe décroissant) :")
for f in hotspots_mc:
    print(f"  {f.relative_path:<20}  CC={f.metrics.mccabe.value:>3} ({f.metrics.mccabe.level})")


## Commentaire humain par fichier

`comment_file` produit un verdict humain avec icône, basé sur le pire niveau des 4 métriques.

In [ ]:
from kodoneko_scanner import comment_file

for f in sorted(report.by_file, key=lambda x: x.relative_path):
    if f.metrics is None:
        continue
    print(f"📄 {f.relative_path}")
    print(f"   {comment_file(f)}")
    print()

## Configuration personnalisée (TOML)

Tous les poids et seuils sont surchargeables via un fichier `kodoneko.toml`. Pratique pour adapter l'outil à un projet spécifique (par ex : durcir les seuils sur un code critique, ou au contraire les détendre sur un POC).

In [ ]:
from kodoneko_scanner import load_config, example_config_toml

# Voir le template complet par défaut
print(example_config_toml()[:1200], "...")

### Exemple : configuration plus stricte sur mccabe

In [ ]:
# Créer une config qui pénalise davantage McCabe
custom_config_path = tmp / "kodoneko.toml"
custom_config_path.write_text('''
[primary_score.weights]
ncloc = 0.05
halstead = 0.05
mccabe = 0.40
understandability = 0.50
''')

custom_cfg = load_config(str(custom_config_path))
print(f"Config chargée : {custom_cfg.source}")
print(f"Poids : {custom_cfg.primary_weights}")
print()

# Re-scanner avec la config personnalisée
report_custom = scan_repo(str(tmp), use_git=False, config=custom_cfg)
bs_custom = compute_primary_quality_score(report_custom, config=custom_cfg)

# Comparaison des deux quality scores
bs_default = compute_primary_quality_score(report)
print(f"Quality score avec poids par défaut       : {bs_default.score:.1f}/100  ({bs_default.grade})")
print(f"Quality score avec poids McCabe-centric   : {bs_custom.score:.1f}/100  ({bs_custom.grade})")


### Recherche automatique du fichier de config

Sans argument explicite, `load_config()` cherche dans cet ordre :

1. Variable d'environnement `KODONEKO_CONFIG`
2. `./kodoneko.toml` (répertoire courant)
3. `~/.config/kodoneko/kodoneko.toml` (utilisateur)
4. Sinon, retourne les défauts (`source="defaults"`)

### Générer un fichier de référence

Depuis la CLI :

```bash
python -m kodoneko_scanner --init-config > kodoneko.toml
```

Cela produit un TOML commenté que tu peux éditer. Cf. README pour le détail de toutes les sections.

## Nettoyage

In [ ]:
import shutil
shutil.rmtree(tmp)
print(f"Dépôt de test supprimé : {tmp}")

## Récapitulatif

| Fonction | Rôle |
|---|---|
| `scan_repo(path, config=...)` | Parcourt le dépôt, retourne `RepoMetrics` |
| `compute_primary_quality_score(report, config=...)` | Calcule le quality score 0-100 + grade A-E |
| `comment_file(file_report)` | Verdict humain par fichier (avec icône) |
| `report.hotspots(top=N, by=...)` | Top N fichiers les plus complexes (par risk) |
| `load_config(path=None)` | Charge `kodoneko.toml` ou les défauts |
| `example_config_toml()` | Retourne un template TOML commenté |

### Rappel essentiel

- `bs.score` : **quality score** (plus c'est haut, mieux c'est).
- `u.risk` : **risk d'understandability** (plus c'est haut, plus c'est dur à comprendre).

### CLI

```bash
python -m kodoneko_scanner /path/to/repo                  # rapport texte
python -m kodoneko_scanner /path/to/repo --json           # rapport JSON
python -m kodoneko_scanner /path/to/repo -c my-config.toml
python -m kodoneko_scanner --init-config > kodoneko.toml
```


---

# Mesurer sur un dépôt réel, et dans le temps

Le scanner réunit toutes les métriques en une vue d'ensemble, couronnée par le
**PrimaryQualityScore** — une note synthétique de 0 à 100 qui condense la santé
d'un dépôt. Cette dernière partie applique le scanner complet à un projet réel,
puis montre comment sa qualité globale évolue dans le temps.

> ⚠️ Cette section clone un dépôt public (nécessite un accès réseau) et, pour
> les langages autres que Python, requiert `tree-sitter`. Sans ces prérequis,
> les cellules affichent un message explicatif et s'interrompent proprement,
> sans erreur.

## 1. Récupérer un dépôt public

Nous clonons un petit projet open-source réel. Le clone est *idempotent* : si le
dépôt est déjà présent localement, on le réutilise sans le retélécharger.

In [ ]:
import subprocess, tempfile
from pathlib import Path

REPO_URL = "https://github.com/gothinkster/django-realworld-example-app.git"
repo_dir = Path(tempfile.gettempdir()) / "kodoneko_demo_repo"

if repo_dir.exists():
    print("Dépôt déjà présent :", repo_dir)
else:
    print("Clonage en cours...")
    res = subprocess.run(
        ["git", "clone", REPO_URL, str(repo_dir)],
        capture_output=True, text=True,
    )
    if res.returncode == 0:
        print("Cloné :", repo_dir)
    else:
        print("Clone impossible (réseau indisponible ?) :")
        print("  ", res.stderr.strip().splitlines()[-1] if res.stderr.strip() else "erreur inconnue")
        repo_dir = None

## 2. Analyser le dépôt complet (instantané)

Premier réflexe : prendre une photographie de l'état actuel. On mesure le PrimaryQualityScore (score de qualité agrégé)
sur l'ensemble du dépôt, tel qu'il est à l'instant présent (le dernier commit).

In [ ]:
from kodoneko_scanner import scan_repo

if repo_dir:
    report = scan_repo(repo_dir)
    print(f"Fichiers analysés : {report.files_scanned}")
    print(f"  NCLOC total              : {report.totals.ncloc}")
    print(f"  Complexité totale        : {report.totals.mccabe_sum}")
    print(f"  Compréhensibilité moyenne: {report.totals.understandability_avg:.1f}/100")
else:
    print("(dépôt indisponible — étape ignorée)")

## 3. Analyser un commit précis

Le moteur temporel `kodoneko_temporal` sait reconstituer l'état du dépôt à
**n'importe quelle référence git** (un tag, une branche, un SHA, ou une
expression comme `HEAD~5`). Ici, on mesure PrimaryQualityScore à ce point de l'histoire, cinq commits
en arrière — un véritable voyage dans le passé du code.

In [ ]:
from kodoneko_temporal import analyze_at_ref

def mesure(path):
    """Applique notre métrique à un état du dépôt."""
    report = scan_repo(path)
    return round(report.totals.understandability_avg, 1)

if repo_dir:
    try:
        point = analyze_at_ref(repo_dir, "HEAD~5", analyzer=mesure)
        print(f"À {point.label} (commit {point.sha[:8]}) : PrimaryQualityScore = {point.result}")
    except Exception as e:
        print(f"Commit indisponible (historique trop court ?) : {e}")
else:
    print("(dépôt indisponible — étape ignorée)")

## 4. Mesurer un écart entre deux moments (*diff temporel*)

La question la plus parlante n'est pas « combien ? » mais « **combien de plus
qu'avant ?** ». En comparant la mesure à deux références, on quantifie ce qu'un
intervalle de développement a réellement produit comme PrimaryQualityScore.

In [ ]:
if repo_dir:
    try:
        avant = analyze_at_ref(repo_dir, "HEAD~10", analyzer=mesure)
        apres = analyze_at_ref(repo_dir, "HEAD", analyzer=mesure)
        delta = apres.result - avant.result
        signe = "+" if delta >= 0 else ""
        # Variation relative en pourcentage (2 décimales), protégée contre la division par zéro
        if avant.result:
            pct = delta / avant.result * 100
            pct_txt = f"{signe}{pct:.2f} %"
        else:
            pct_txt = "n/a (valeur initiale nulle)"
        print(f"PrimaryQualityScore il y a 10 commits : {avant.result}")
        print(f"PrimaryQualityScore aujourd'hui       : {apres.result}")
        print(f"Variation absolue    : {signe}{delta}")
        print(f"Variation relative   : {pct_txt}")
    except Exception as e:
        print(f"Diff indisponible : {e}")
else:
    print("(dépôt indisponible — étape ignorée)")

### La qualité globale, mois après mois

Un score unique ne remplace pas le détail, mais il donne le pouls du projet.
Suivre le PrimaryQualityScore dans le temps, c'est disposer d'un
électrocardiogramme de la qualité : on voit les phases saines, les coups de
fatigue, et l'effet des grands nettoyages.

In [ ]:
from kodoneko_temporal import analyze_over_windows

if repo_dir:
    serie = analyze_over_windows(repo_dir, analyzer=mesure, strategy="monthly")
    print(f"PrimaryQualityScore à la fin de chaque mois ({len(serie)} fenêtres) :\n")
    maxv = max((p.result for p in serie), default=1) or 1
    precedent = None
    for point in serie:
        barre = "█" * min(40, int(point.result / maxv * 40))
        # Variation par rapport au mois précédent, en % (2 décimales)
        if precedent is None:
            var_txt = "     —  "
        elif precedent:
            pct = (point.result - precedent) / precedent * 100
            var_txt = f"{pct:+6.2f} %"
        else:
            var_txt = "    n/a "
        print(f"  {point.label}  {point.result:>8}  {var_txt}  {barre}")
        precedent = point.result
else:
    print("(dépôt indisponible — étape ignorée)")

## En résumé

Nous venons de parcourir les quatre façons d'interroger le PrimaryQualityScore (score de qualité agrégé) :

1. **L'instantané** — l'état présent du dépôt entier.
2. **Le commit précis** — la mesure à un point quelconque de l'histoire.
3. **Le diff temporel** — l'écart entre deux moments, qui isole ce qu'une
   période a produit.
4. **La série temporelle** — la trajectoire complète, qui révèle les tendances.

Le même analyseur (`mesure`) a servi pour les quatre : c'est toute la force du
moteur temporel, qui découple *ce qu'on mesure* de *quand on le mesure*.